# Pitch gate sweep — equally weighted Bach10, URMP and Coco

Reuse all completed **44.1 kHz** Coco candidate scores, raw pitch evidence and per-volume HMM tracks from `appendix_195_44100_v2`. Migration validates source hashes, grids, package versions and algorithm identity; incompatible caches are rejected. Only the 17 new stems need pitch inference. The new dataset-balanced winner is exported for all 212 stems, even if it differs from the old Coco winner.

Sweep completed-audio `posthoc_min_volume` (whole-track p95 RMS fraction) and `posthoc_unv_thresh`. Live gates and HMM confidence stay fixed. The volume gate affects HMM regions, so tracking is cached per volume. Rank by dataset-macro Overall Accuracy; also report RPA, RCA, voicing recall and false alarm. Scoring uses the normal benchmark 10 ms reference grid, with 44.1 kHz analysis and a 4096/128 frame/hop.


In [1]:
from pathlib import Path
import os, sys
# Limit native threads before importing numerical libraries: 8 processes × 1 thread.
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ[name] = '1'
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiments.modules.sweeps import AppendixSweeps as sweep
import importlib
sweep = importlib.reload(sweep)
import pandas as pd
from experiments.paths import relocated_path
RUN_SWEEPS = False  # Set True with a NEW OUTPUT directory for new experiments.
WORKERS = 8
OUTPUT = ROOT / 'experiments/results/caches/sweeps/balanced_212_v3'
DATASET_ROOT = ROOT / 'experiments/datasets/coco/cocochorales_tiny'
SPLIT = 'test'  # Locally available partition; NEW development cohort, NOT a test claim.
SEED = 20261116


## Frozen cohort and weighting

Pitch and note tuning use **212 stems: 195 existing CocoChorales + 4 Bach10 + 13 URMP**. Each real dataset contributes one stem randomly sampled within each instrument, with seed 20261116. The manifest freezes the selection; restarting does not resample. All 195 Coco stems remain unchanged. These are development samples, not an untouched test set.

Pool metric counts within each dataset, then take the arithmetic mean of the three dataset metrics (one-third each). This is not a pooled 212-track score. `by_dataset.csv` retains individual dataset scores; count columns in aggregate reports are summed counts and do not reconstruct the dataset-macro F1.

Run notebooks in order: **pitch → note → mistake**. The mistake stage uses only the 195 Coco sources, with the selected pitch/note settings. Production defaults and prior output directories remain unchanged.


In [2]:
# Freeze the sample and validate source files; no detector is run here.
manifest = (sweep.materialize(OUTPUT, root=DATASET_ROOT, split=SPLIT,
                              seed=SEED, workers=WORKERS) if RUN_SWEEPS
            else sweep.read(OUTPUT / 'cohort.json'))
cohort = pd.DataFrame(manifest['items'])
assert cohort.groupby('dataset').size().to_dict() == {'bach10-original': 4, 'coco': 195, 'urmp': 13}
display(pd.crosstab(cohort.instrument, cohort.dataset))
print('Frozen manifest:', OUTPUT / 'cohort.json')


dataset,bach10-original,coco,urmp
instrument,,,
bassoon,1,15,1
cello,0,15,1
clarinet,1,15,1
double bass,0,15,1
flute,0,15,1
horn,0,15,1
oboe,0,15,1
saxophone,1,15,1
trombone,0,15,1


Frozen manifest: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/cohort.json


## Prespecified parameter grid

In [3]:
PITCH_AXES = dict(sweep.PITCH_AXES)
# Volume gates: 0, 0.02, 0.04, 0.05, 0.06, 0.08 (36 total candidates).
sweep.result_table(pd.DataFrame([v['params'] for v in sweep.grid(PITCH_AXES)]))


,Min Volume,Unv Thresh
0,0.00,0.9500
1,0.00,0.9700
2,0.00,0.9800
3,0.00,0.9850
4,0.00,0.9875
5,0.00,0.9900
6,0.02,0.9500
7,0.02,0.9700
8,0.02,0.9800
9,0.02,0.9850


## Run and rank

In [4]:
# Reuse completed Coco scores; compute the 17 added stems and export the new winner.
summary = (sweep.run_pitch(OUTPUT, axes=PITCH_AXES, workers=WORKERS) if RUN_SWEEPS
           else pd.read_csv(OUTPUT / 'pitch/summary.csv'))
display(summary.head(15))
print('Winner:', OUTPUT / 'pitch/best.json')
print('Per-dataset results:', OUTPUT / 'pitch/by_dataset.csv')


212/212 | test__woodwind_track229681__1_flute | Min Volume=0.06, Unv Thresh=0.9875 | Caching winner

,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Min Volume,Unv Thresh
0,0.958329,0.964595,0.939673,0.978877,0.197039,0.06,0.9875
1,0.957225,0.963445,0.939501,0.977305,0.192630,0.06,0.9850
2,0.959212,0.965436,0.939371,0.979788,0.203660,0.05,0.9875
3,0.958103,0.964282,0.939229,0.978212,0.199060,0.05,0.9850
4,0.959891,0.966157,0.938881,0.980543,0.211653,0.04,0.9875
5,0.954818,0.960957,0.938770,0.974276,0.185728,0.06,0.9800
6,0.958780,0.965000,0.938757,0.978963,0.206878,0.04,0.9850
7,0.955044,0.961303,0.938754,0.975430,0.185826,0.08,0.9875
8,0.955686,0.961783,0.938556,0.975172,0.191731,0.05,0.9800
9,0.953958,0.960172,0.938489,0.973884,0.181961,0.08,0.9850


Winner: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/pitch/best.json
Per-dataset results: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/pitch/by_dataset.csv


## Resume and provenance

`contract.json` records sources, code, packages and grid. `reuse.json` records the historical Coco source. `rows.csv`, `summary.csv`, `by_dataset.csv`, and `best.json` retain all scores and the selected artifacts. Reruns reuse completed checkpoints. Source or algorithm changes require a new output directory. The old native-16-kHz run is never imported.

Adding 0.05 preserves completed candidates and raw pitch caches. Rerun the pitch notebook to score the six new combinations; prior reports are archived before selecting the expanded-grid winner.
